# A4 - Module A evaluation

Multi-scale LSTM load forecaster (CQR-calibrated) evaluated on 2024 val and 2025 Q1 test.

**Sections**
1. Setup & data loading
2. Sample week - predictions vs reality
3. Point accuracy vs baselines (MAE / RMSE / MAPE)
4. Probabilistic calibration
5. Error distribution
6. By forecast horizon (h1-h24)
7. Segment breakdowns
8. Diebold-Mariano significance
9. Summary leaderboard

In [ ]:
import sys, pickle
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

CKPT      = REPO_ROOT / 'checkpoints' / 'module_a' / 'best.pt'
CQR_STATE = REPO_ROOT / 'checkpoints' / 'module_a' / 'cqr_state.pkl'

## 1. Load data, checkpoint, CQR state

In [ ]:
from data.loaders import load_val, load_test
from module_a.features import ALL_BUNDLES, TARGET_COL, build_features
from module_a.model import MultiScaleLSTMForecaster
from module_a.evaluation import (
    mae, rmse, mape, nmae, pinball, coverage,
    mean_interval_width, winkler_score,
    empirical_coverage_at_quantiles, diebold_mariano,
)

forecaster = MultiScaleLSTMForecaster.load(CKPT)

with open(CQR_STATE, 'rb') as f:
    cqr = pickle.load(f)
delta = cqr['delta']
print(f'CQR delta = {delta:+.1f} MW')

val_raw  = load_val()
test_raw = load_test()

bundles = list(ALL_BUNDLES)
val_feat  = build_features(val_raw,  bundles=bundles)
test_feat = build_features(test_raw, bundles=bundles)

val_actual  = val_raw[TARGET_COL]
test_actual = test_raw[TARGET_COL]

print(f'Val:  {val_actual.index[0]}  -  {val_actual.index[-1]}')
print(f'Test: {test_actual.index[0]}  -  {test_actual.index[-1]}')

In [ ]:
def predict_calibrated(forecaster, feat, delta):
    """predict_quantiles + CQR shift on q10/q90."""
    long = forecaster.predict_quantiles(feat, stride=1)
    long = long.copy()
    long['q10'] = long['q10'] - delta
    long['q90'] = long['q90'] + delta
    return long

def align(long_preds, actual):
    """Flatten (origin, h) → target_ts rows with actual values."""
    rows = []
    for (origin, h), row in long_preds.iterrows():
        target_ts = origin + pd.Timedelta(hours=int(h))
        if target_ts in actual.index:
            rows.append({'target_ts': target_ts, 'origin': origin, 'h': h,
                         'q10': row['q10'], 'q50': row['q50'],
                         'q90': row['q90'], 'y': actual[target_ts]})
    df = pd.DataFrame(rows)
    df['error'] = df['q50'] - df['y']
    df['abs_error'] = df['error'].abs()
    return df

print('Predicting val...')
val_preds = predict_calibrated(forecaster, val_feat, delta)
val_df    = align(val_preds, val_actual)

print('Predicting test...')
test_preds = predict_calibrated(forecaster, test_feat, delta)
test_df    = align(test_preds, test_actual)

print(f'Val rows: {len(val_df):,}   Test rows: {len(test_df):,}')

## 2. Sample week - predictions vs reality

One winter week (Jan 2025) and one summer week (Jul 2024) showing q10/q50/q90 vs actual load.
Predictions are the h=1 step from each origin (true 1-step-ahead).

In [ ]:
def plot_sample_week(df, title, ax):
    """Plot one week of h=1 predictions vs actual."""
    h1 = df[df['h'] == 1].set_index('target_ts').sort_index()
    ax.fill_between(h1.index, h1['q10'], h1['q90'],
                    alpha=0.25, color='steelblue', label='80% PI')
    ax.plot(h1.index, h1['q50'], color='steelblue', lw=1.5, label='q50 forecast')
    ax.plot(h1.index, h1['y'],   color='black', lw=1.2, ls='--', label='Actual')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Load (MW)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %d'))
    ax.legend(fontsize=8)

# Winter week: first week of Jan 2025 test set
jan_start = pd.Timestamp('2025-01-06', tz='UTC')
jan_end   = jan_start + pd.Timedelta(days=7)
test_jan  = test_df[(test_df['target_ts'] >= jan_start) & (test_df['target_ts'] < jan_end)]

# Summer week: first week of Jul 2024 val set
jul_start = pd.Timestamp('2024-07-01', tz='UTC')
jul_end   = jul_start + pd.Timedelta(days=7)
val_jul   = val_df[(val_df['target_ts'] >= jul_start) & (val_df['target_ts'] < jul_end)]

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)
plot_sample_week(val_jul,  'Summer week - Jul 2024 (val)', axes[0])
plot_sample_week(test_jan, 'Winter week - Jan 2025 (test)', axes[1])
fig.suptitle('Module A: MultiScaleLSTM load forecasts (h=1, CQR-calibrated)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'module_a_sample_weeks.png', bbox_inches='tight')
plt.show()

## 3. Point accuracy vs baselines

In [ ]:
def baselines(df, actual):
    """Return naive-24h and seasonal-naive-168h predictions aligned to df['target_ts']."""
    naive24  = actual.shift(24).reindex(df['target_ts']).values
    naive168 = actual.shift(168).reindex(df['target_ts']).values
    return naive24, naive168

def point_table(df, actual, label):
    yt = df['y'].values
    yp = df['q50'].values
    n24, n168 = baselines(df, actual)
    mask = ~np.isnan(n24) & ~np.isnan(n168)
    rows = []
    for name, pred in [('MultiScaleLSTM', yp), ('Naive-24h', n24), ('SeasonalNaive-168h', n168)]:
        m = mask if name != 'MultiScaleLSTM' else np.ones(len(yt), bool)
        yt_m = yt[m]; p_m = pred[m]
        rows.append({'Model': name,
                     'MAE': round(mae(yt_m, p_m), 0),
                     'RMSE': round(rmse(yt_m, p_m), 0),
                     'MAPE%': round(mape(yt_m, p_m)*100, 2),
                     'nMAE%': round(nmae(yt_m, p_m)*100, 2)})
    return pd.DataFrame(rows).set_index('Model')

print('=== VAL 2024 ===')
pt_val = point_table(val_df, val_actual, 'val')
display(pt_val)

print('\n=== TEST 2025 Q1 ===')
pt_test = point_table(test_df, test_actual, 'test')
display(pt_test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, pt, split in zip(axes, [pt_val, pt_test], ['Val 2024', 'Test 2025 Q1']):
    colors = ['steelblue', 'grey', 'lightcoral']
    bars = ax.bar(pt.index, pt['MAE'], color=colors, edgecolor='white')
    ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
    ax.set_title(f'MAE by model - {split}', fontsize=11)
    ax.set_ylabel('MAE (MW)')
    ax.set_ylim(0, pt['MAE'].max() * 1.2)
    ax.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'module_a_mae_comparison.png', bbox_inches='tight')
plt.show()

## 4. Probabilistic calibration

In [ ]:
def prob_table(df, label):
    yt  = df['y'].values
    q10 = df['q10'].values
    q50 = df['q50'].values
    q90 = df['q90'].values
    calib = empirical_coverage_at_quantiles(yt, q10, q50, q90)
    return pd.DataFrame([{
        'split':        label,
        'pinball_q10':  round(pinball(yt, q10, 0.1), 1),
        'pinball_q50':  round(pinball(yt, q50, 0.5), 1),
        'pinball_q90':  round(pinball(yt, q90, 0.9), 1),
        'coverage_80%': round(coverage(yt, q10, q90)*100, 1),
        'width_MW':     round(mean_interval_width(q10, q90), 0),
        'winkler':      round(winkler_score(yt, q10, q90), 0),
        'calib_q10':    round(calib['below_q10']*100, 1),
        'calib_q50':    round(calib['below_q50']*100, 1),
        'calib_q90':    round(calib['below_q90']*100, 1),
    }])

prob = pd.concat([prob_table(val_df, 'val'), prob_table(test_df, 'test')]).set_index('split')
display(prob)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, df, actual, split in zip(axes,
                                  [val_df, test_df],
                                  [val_actual, test_actual],
                                  ['Val 2024', 'Test 2025 Q1']):
    yt  = df['y'].values
    q10 = df['q10'].values
    q90 = df['q90'].values
    calib = empirical_coverage_at_quantiles(yt, q10, df['q50'].values, q90)

    nominal  = [10, 50, 90]
    empirical = [
        calib['below_q10']*100,
        calib['below_q50']*100,
        calib['below_q90']*100,
    ]
    x = np.arange(3)
    w = 0.35
    ax.bar(x - w/2, nominal,  w, label='Nominal', color='lightgrey', edgecolor='grey')
    ax.bar(x + w/2, empirical, w, label='Empirical', color='steelblue', edgecolor='navy')
    ax.set_xticks(x)
    ax.set_xticklabels(['q10', 'q50', 'q90'])
    ax.axhline(10, color='red', ls=':', lw=0.8)
    ax.axhline(50, color='red', ls=':', lw=0.8)
    ax.axhline(90, color='red', ls=':', lw=0.8)
    ax.set_ylabel('% actuals below quantile')
    ax.set_title(f'Calibration - {split}', fontsize=11)
    ax.set_ylim(0, 105)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'module_a_calibration.png', bbox_inches='tight')
plt.show()

In [ ]:
# Reliability diagram: predicted interval coverage across quantile levels
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, df, split in zip(axes, [val_df, test_df], ['Val 2024', 'Test 2025 Q1']):
    yt  = df['y'].values
    q50 = df['q50'].values
    # Use the pre-calibrated q50 ± scaled widths to simulate quantiles
    q10 = df['q10'].values
    q90 = df['q90'].values
    half_width = (q90 - q10) / 2

    alphas = np.linspace(0.02, 0.98, 49)
    coverages = []
    for a in alphas:
        lo = q50 - np.quantile(half_width, 1 - a) * (half_width / np.mean(half_width))
        hi = q50 + np.quantile(half_width, 1 - a) * (half_width / np.mean(half_width))
        coverages.append(np.mean((yt >= lo) & (yt <= hi)))

    ax.plot(alphas, coverages, color='steelblue', lw=2, label='Model')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')
    ax.fill_between(alphas, alphas, coverages,
                    where=np.array(coverages) < alphas,
                    alpha=0.15, color='red', label='Under-coverage')
    ax.fill_between(alphas, alphas, coverages,
                    where=np.array(coverages) > alphas,
                    alpha=0.15, color='green', label='Over-coverage')
    ax.set_xlabel('Nominal coverage')
    ax.set_ylabel('Empirical coverage')
    ax.set_title(f'Coverage reliability - {split}', fontsize=11)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'module_a_reliability.png', bbox_inches='tight')
plt.show()

## 5. Error distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, df, split in zip(axes, [val_df, test_df], ['Val 2024', 'Test 2025 Q1']):
    errs = df['error'].values
    ax.hist(errs, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(0, color='black', lw=1.2, ls='--')
    ax.axvline(np.mean(errs), color='red', lw=1.5, label=f'Mean bias={np.mean(errs):.0f} MW')
    ax.axvline(np.median(errs), color='orange', lw=1.5, ls='--',
               label=f'Median={np.median(errs):.0f} MW')
    ax.set_xlabel('q50 forecast error (MW)  [positive = overforecast]')
    ax.set_ylabel('Count')
    ax.set_title(f'Error distribution - {split}', fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'module_a_error_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# Actual vs predicted scatter
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, df, split in zip(axes, [val_df, test_df], ['Val 2024', 'Test 2025 Q1']):
    # Subsample for speed
    sample = df.sample(min(5000, len(df)), random_state=0)
    ax.scatter(sample['y'], sample['q50'], alpha=0.15, s=4, color='steelblue')
    lo, hi = df['y'].min(), df['y'].max()
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1, label='y=x')
    ax.set_xlabel('Actual load (MW)')
    ax.set_ylabel('Predicted q50 (MW)')
    ax.set_title(f'Actual vs Predicted - {split}', fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'module_a_scatter.png', bbox_inches='tight')
plt.show()

## 6. By forecast horizon (h1-h24)

In [ ]:
def by_horizon(df, actual, lag):
    """MAE by horizon h; baseline is naive with given lag."""
    naive = actual.shift(lag).reindex(df['target_ts']).values
    rows = []
    for h in range(1, 25):
        sub = df[df['h'] == h]
        n   = naive[df['h'] == h]
        mask = ~np.isnan(n)
        rows.append({
            'h': h,
            'MAE_model':    mae(sub['y'].values, sub['q50'].values),
            'MAE_naive168': mae(sub['y'].values[mask], n[mask]),
            'coverage_80':  coverage(sub['y'].values, sub['q10'].values, sub['q90'].values) * 100,
            'width':        mean_interval_width(sub['q10'].values, sub['q90'].values),
        })
    return pd.DataFrame(rows).set_index('h')

bh_val  = by_horizon(val_df,  val_actual,  168)
bh_test = by_horizon(test_df, test_actual, 168)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for col, (bh, split) in enumerate([(bh_val, 'Val 2024'), (bh_test, 'Test 2025 Q1')]):
    # MAE by horizon
    ax = axes[0, col]
    ax.plot(bh.index, bh['MAE_model'],    color='steelblue', lw=2, marker='o', ms=4, label='MultiScaleLSTM')
    ax.plot(bh.index, bh['MAE_naive168'], color='grey',      lw=1.5, ls='--', label='SeasonalNaive-168h')
    ax.set_xlabel('Forecast horizon (h)')
    ax.set_ylabel('MAE (MW)')
    ax.set_title(f'MAE by horizon - {split}', fontsize=10)
    ax.set_xticks(range(1, 25, 2))
    ax.legend(fontsize=8)

    # Coverage by horizon
    ax = axes[1, col]
    ax.plot(bh.index, bh['coverage_80'], color='steelblue', lw=2, marker='o', ms=4)
    ax.axhline(80, color='red', ls='--', lw=1.2, label='Target 80%')
    ax.set_xlabel('Forecast horizon (h)')
    ax.set_ylabel('Coverage 80% PI (%)')
    ax.set_title(f'Coverage by horizon - {split}', fontsize=10)
    ax.set_xticks(range(1, 25, 2))
    ax.set_ylim(50, 100)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'module_a_by_horizon.png', bbox_inches='tight')
plt.show()

## 7. Segment breakdowns

In [ ]:
def season(ts): 
    m = ts.month
    return {12:'winter',1:'winter',2:'winter',3:'spring',4:'spring',5:'spring',
            6:'summer',7:'summer',8:'summer',9:'autumn',10:'autumn',11:'autumn'}[m]

def hour_group(h):
    if 6 <= h <= 9:   return 'morning_peak'
    if 10 <= h <= 16: return 'midday'
    if 17 <= h <= 20: return 'evening_peak'
    return 'night'

def add_segments(df):
    df = df.copy()
    df['season']    = df['target_ts'].dt.tz_convert('Europe/Berlin').apply(season)
    df['day_type']  = df['target_ts'].dt.dayofweek.apply(lambda d: 'weekend' if d >= 5 else 'weekday')
    df['hour_group'] = df['target_ts'].dt.tz_convert('Europe/Berlin').dt.hour.apply(hour_group)
    return df

val_seg  = add_segments(val_df)
test_seg = add_segments(test_df)

def seg_summary(df, col):
    return (df.groupby(col)
              .apply(lambda g: pd.Series({
                  'MAE': mae(g['y'], g['q50']),
                  'nMAE%': nmae(g['y'], g['q50']) * 100,
                  'cov80': coverage(g['y'], g['q10'], g['q90']) * 100,
              }), include_groups=False)
              .round(1))

for col in ['season', 'day_type', 'hour_group']:
    print(f'\n--- {col} ---')
    print('VAL:', seg_summary(val_seg, col).to_string())
    print('TEST:', seg_summary(test_seg, col).to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

segments = ['season', 'day_type', 'hour_group']

for col_i, (seg_col, seg_label) in enumerate([
    ('season', 'Season'), ('day_type', 'Day type'), ('hour_group', 'Hour group')
]):
    for row_i, (seg_df, split) in enumerate([(val_seg, 'Val 2024'), (test_seg, 'Test 2025 Q1')]):
        ax = axes[row_i, col_i]
        sm = seg_summary(seg_df, seg_col)
        bars = ax.bar(sm.index, sm['MAE'], color='steelblue', edgecolor='white')
        ax.bar_label(bars, fmt='%.0f', padding=2, fontsize=8)
        ax.set_title(f'{seg_label} - {split}', fontsize=9)
        ax.set_ylabel('MAE (MW)' if col_i == 0 else '')
        ax.tick_params(axis='x', labelsize=8, rotation=15)
        ax.set_ylim(0, sm['MAE'].max() * 1.2)

plt.suptitle('MAE by segment - MultiScaleLSTM (CQR-calibrated)', fontsize=11)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'reports' / 'module_a_segments.png', bbox_inches='tight')
plt.show()

## 8. Diebold-Mariano significance

In [ ]:
def dm_table(df, actual):
    yt  = df['y'].values
    yp  = df['q50'].values
    n24  = actual.shift(24).reindex(df['target_ts']).values
    n168 = actual.shift(168).reindex(df['target_ts']).values
    mask = ~np.isnan(n24) & ~np.isnan(n168)

    e_model   = np.abs(yt[mask] - yp[mask])
    e_naive   = np.abs(yt[mask] - n24[mask])
    e_snaive  = np.abs(yt[mask] - n168[mask])

    rows = []
    for name, e_base in [('vs Naive-24h', e_naive), ('vs SeasonalNaive-168h', e_snaive)]:
        dm_stat, dm_p = diebold_mariano(e_model, e_base)
        rows.append({'Comparison': name,
                     'DM stat': round(dm_stat, 3),
                     'p-value': round(dm_p, 4),
                     'Significant': '*' if dm_p < 0.05 else '',
                     'Winner': 'LSTM' if dm_stat < 0 else 'Baseline'})
    return pd.DataFrame(rows).set_index('Comparison')

print('=== VAL ===')
display(dm_table(val_df, val_actual))
print('\n=== TEST ===')
display(dm_table(test_df, test_actual))

## 9. Summary leaderboard

In [ ]:
def full_leaderboard(df, actual):
    yt  = df['y'].values
    q10 = df['q10'].values
    q50 = df['q50'].values
    q90 = df['q90'].values
    n24  = actual.shift(24).reindex(df['target_ts']).values
    n168 = actual.shift(168).reindex(df['target_ts']).values
    mask = ~np.isnan(n24) & ~np.isnan(n168)
    calib = empirical_coverage_at_quantiles(yt, q10, q50, q90)

    rows = [
        {'Model': 'MultiScaleLSTM+CQR',
         'MAE': round(mae(yt, q50), 0),
         'RMSE': round(rmse(yt, q50), 0),
         'nMAE%': round(nmae(yt, q50)*100, 2),
         'pinball_q10': round(pinball(yt, q10, 0.1), 0),
         'pinball_q50': round(pinball(yt, q50, 0.5), 0),
         'pinball_q90': round(pinball(yt, q90, 0.9), 0),
         'cov_80%': round(coverage(yt, q10, q90)*100, 1),
         'width_MW': round(mean_interval_width(q10, q90), 0),
         'calib_q10': round(calib['below_q10']*100, 1),
         'calib_q90': round(calib['below_q90']*100, 1),
        },
        {'Model': 'Naive-24h',
         'MAE': round(mae(yt[mask], n24[mask]), 0),
         'RMSE': round(rmse(yt[mask], n24[mask]), 0),
         'nMAE%': round(nmae(yt[mask], n24[mask])*100, 2),
         **{k: '-' for k in ['pinball_q10','pinball_q50','pinball_q90','cov_80%','width_MW','calib_q10','calib_q90']}
        },
        {'Model': 'SeasonalNaive-168h',
         'MAE': round(mae(yt[mask], n168[mask]), 0),
         'RMSE': round(rmse(yt[mask], n168[mask]), 0),
         'nMAE%': round(nmae(yt[mask], n168[mask])*100, 2),
         **{k: '-' for k in ['pinball_q10','pinball_q50','pinball_q90','cov_80%','width_MW','calib_q10','calib_q90']}
        },
    ]
    return pd.DataFrame(rows).set_index('Model')

print('=== VAL 2024 - LEADERBOARD ===')
lb_val = full_leaderboard(val_df, val_actual)
display(lb_val)

print('\n=== TEST 2025 Q1 - LEADERBOARD ===')
lb_test = full_leaderboard(test_df, test_actual)
display(lb_test)

print('\nModel checkpoint:', CKPT)
print('CQR delta:',delta,'MW')